In [ ]:
from itertools import product
from pathlib import Path

import numpy as np
import pandas as pd

from scipy.stats import rankdata
from scipy.stats import wilcoxon

In [ ]:
EXPECTED_SPLIT_SEEDS = [
    13,
    21,
    40,
    42,
    73,
    101,
]

EXPECTED_MODEL_SEED = 40

CONDITIONS = [
    "pair_controlled",
    "max_cross_split",
]

MODELS = [
    "bertimbau",
    "ensemble",
]

DESCRIPTIVE_METRICS = [
    "accuracy",
    "precision_non_pun",
    "recall_non_pun",
    "f1_non_pun",
    "precision_pun",
    "recall_pun",
    "f1_pun",
    "precision_macro",
    "recall_macro",
    "f1_macro",
    "precision_weighted",
    "recall_weighted",
    "f1_weighted",
]

INFERENTIAL_METRICS = [
    "accuracy",
    "f1_macro",
]

BOOTSTRAP_RESAMPLES = 50000
STATISTICAL_SEED = 2026

In [ ]:
def find_project_root(
    start_path=None,
):
    current = Path(
        start_path
        or Path.cwd()
    ).resolve()

    while True:
        if (
            current
            / "results"
            / "bertimbau"
        ).is_dir():
            return current

        if current == current.parent:
            break

        current = current.parent

    raise FileNotFoundError(
        "Could not locate the project root."
    )


PROJECT_ROOT = find_project_root()

RESULTS_DIR = (
    PROJECT_ROOT
    / "results"
)

OUTPUT_PATH = (
    RESULTS_DIR
    / "statistical_analysis.csv"
)

print(
    "Project root:",
    PROJECT_ROOT,
)

print(
    "Output:",
    OUTPUT_PATH,
)

In [ ]:
def load_condition_results(
    model,
    condition,
):
    path = (
        RESULTS_DIR
        / model
        / condition
        / "runs.csv"
    )

    if not path.is_file():
        raise FileNotFoundError(
            f"Missing results file: {path}"
        )

    dataframe = pd.read_csv(
        path
    )

    required_columns = {
        "condition",
        "split_seed",
        "model_seed",
        *DESCRIPTIVE_METRICS,
    }

    missing_columns = (
        required_columns
        - set(
            dataframe.columns
        )
    )

    if missing_columns:
        raise ValueError(
            f"{model}/{condition}: missing columns "
            f"{sorted(missing_columns)}."
        )

    if len(dataframe) != len(
        EXPECTED_SPLIT_SEEDS
    ):
        raise ValueError(
            f"{model}/{condition}: expected "
            f"{len(EXPECTED_SPLIT_SEEDS)} runs, "
            f"found {len(dataframe)}."
        )

    if dataframe[
        "split_seed"
    ].duplicated().any():
        raise ValueError(
            f"{model}/{condition}: duplicated split seeds."
        )

    observed_split_seeds = sorted(
        dataframe[
            "split_seed"
        ]
        .astype(int)
        .tolist()
    )

    expected_split_seeds = sorted(
        EXPECTED_SPLIT_SEEDS
    )

    if (
        observed_split_seeds
        != expected_split_seeds
    ):
        raise ValueError(
            f"{model}/{condition}: expected split seeds "
            f"{expected_split_seeds}, found "
            f"{observed_split_seeds}."
        )

    observed_conditions = set(
        dataframe[
            "condition"
        ]
        .astype(str)
    )

    if observed_conditions != {
        condition
    }:
        raise ValueError(
            f"{model}/{condition}: unexpected condition "
            f"values {sorted(observed_conditions)}."
        )

    observed_model_seeds = set(
        dataframe[
            "model_seed"
        ]
        .astype(int)
    )

    if observed_model_seeds != {
        EXPECTED_MODEL_SEED
    }:
        raise ValueError(
            f"{model}/{condition}: expected model seed "
            f"{EXPECTED_MODEL_SEED}, found "
            f"{sorted(observed_model_seeds)}."
        )

    for metric in DESCRIPTIVE_METRICS:
        if dataframe[
            metric
        ].isna().any():
            raise ValueError(
                f"{model}/{condition}: metric "
                f"{metric} contains missing values."
            )

        values = dataframe[
            metric
        ].astype(float)

        if not (
            values.between(
                0.0,
                1.0,
                inclusive="both",
            )
        ).all():
            raise ValueError(
                f"{model}/{condition}: metric "
                f"{metric} contains values outside [0, 1]."
            )

    return (
        dataframe
        .sort_values(
            "split_seed"
        )
        .reset_index(
            drop=True
        )
    )


results_by_model = {}

for model in MODELS:
    results_by_model[
        model
    ] = {}

    for condition in CONDITIONS:
        results_by_model[
            model
        ][
            condition
        ] = load_condition_results(
            model=model,
            condition=condition,
        )

print(
    "All result files were loaded and validated."
)

In [ ]:
for model in MODELS:
    pair_controlled_df = (
        results_by_model[
            model
        ][
            "pair_controlled"
        ]
    )

    max_cross_split_df = (
        results_by_model[
            model
        ][
            "max_cross_split"
        ]
    )

    pair_controlled_seeds = (
        pair_controlled_df[
            "split_seed"
        ]
        .astype(int)
        .to_numpy()
    )

    max_cross_split_seeds = (
        max_cross_split_df[
            "split_seed"
        ]
        .astype(int)
        .to_numpy()
    )

    if not np.array_equal(
        pair_controlled_seeds,
        max_cross_split_seeds,
    ):
        raise ValueError(
            f"{model}: split seeds are not aligned "
            "between conditions."
        )

print(
    "Paired split seeds were validated."
)

In [ ]:
descriptive_rows = []

for model in MODELS:
    for condition in CONDITIONS:
        dataframe = (
            results_by_model[
                model
            ][
                condition
            ]
        )

        for metric in DESCRIPTIVE_METRICS:
            values = (
                dataframe[
                    metric
                ]
                .astype(float)
            )

            mean_value = float(
                values.mean()
            )

            std_value = float(
                values.std(
                    ddof=1
                )
            )

            descriptive_rows.append(
                {
                    "model": model,
                    "condition": condition,
                    "metric": metric,
                    "runs": int(
                        len(
                            values
                        )
                    ),
                    "mean": mean_value,
                    "std": std_value,
                    "mean_std": (
                        f"{mean_value:.4f} "
                        f"± {std_value:.4f}"
                    ),
                }
            )

descriptive_summary_df = pd.DataFrame(
    descriptive_rows
)

display(
    descriptive_summary_df
)

In [ ]:
display(
    descriptive_summary_df.loc[
        descriptive_summary_df[
            "metric"
        ]
        == "accuracy"
    ]
)

In [ ]:
class_metrics = [
    "precision_non_pun",
    "recall_non_pun",
    "f1_non_pun",
    "precision_pun",
    "recall_pun",
    "f1_pun",
]

display(
    descriptive_summary_df.loc[
        descriptive_summary_df[
            "metric"
        ].isin(
            class_metrics
        )
    ]
)

In [ ]:
def exact_paired_permutation_test(
    differences,
):
    differences = np.asarray(
        differences,
        dtype=float,
    )

    if differences.ndim != 1:
        raise ValueError(
            "Differences must be one-dimensional."
        )

    if len(differences) == 0:
        raise ValueError(
            "Differences cannot be empty."
        )

    observed_difference = float(
        differences.mean()
    )

    sign_patterns = np.asarray(
        list(
            product(
                [-1.0, 1.0],
                repeat=len(
                    differences
                ),
            )
        ),
        dtype=float,
    )

    permuted_differences = (
        sign_patterns
        * differences
    ).mean(
        axis=1
    )

    tolerance = (
        np.finfo(float).eps
        * 100
    )

    extreme_count = int(
        (
            np.abs(
                permuted_differences
            )
            >= (
                abs(
                    observed_difference
                )
                - tolerance
            )
        ).sum()
    )

    p_value = float(
        extreme_count
        / len(
            permuted_differences
        )
    )

    return {
        "permutation_statistic": observed_difference,
        "permutation_p_value": p_value,
        "permutation_count": int(
            len(
                permuted_differences
            )
        ),
    }

In [ ]:
def paired_bootstrap_interval(
    differences,
    confidence_level=0.95,
    resamples=BOOTSTRAP_RESAMPLES,
    random_seed=STATISTICAL_SEED,
):
    differences = np.asarray(
        differences,
        dtype=float,
    )

    if differences.ndim != 1:
        raise ValueError(
            "Differences must be one-dimensional."
        )

    if len(differences) < 2:
        raise ValueError(
            "At least two differences are required."
        )

    rng = np.random.default_rng(
        random_seed
    )

    sample_indices = rng.integers(
        low=0,
        high=len(
            differences
        ),
        size=(
            resamples,
            len(
                differences
            ),
        ),
    )

    bootstrap_means = (
        differences[
            sample_indices
        ]
        .mean(
            axis=1
        )
    )

    alpha = (
        1.0
        - confidence_level
    )

    lower = float(
        np.quantile(
            bootstrap_means,
            alpha / 2.0,
        )
    )

    upper = float(
        np.quantile(
            bootstrap_means,
            1.0 - alpha / 2.0,
        )
    )

    return {
        "bootstrap_confidence_level": float(
            confidence_level
        ),
        "bootstrap_resamples": int(
            resamples
        ),
        "bootstrap_ci_low": lower,
        "bootstrap_ci_high": upper,
    }

In [ ]:
def paired_cohens_dz(
    differences,
):
    differences = np.asarray(
        differences,
        dtype=float,
    )

    standard_deviation = float(
        differences.std(
            ddof=1
        )
    )

    if np.isclose(
        standard_deviation,
        0.0,
    ):
        if np.isclose(
            differences.mean(),
            0.0,
        ):
            return 0.0

        return float(
            np.sign(
                differences.mean()
            )
            * np.inf
        )

    return float(
        differences.mean()
        / standard_deviation
    )


def matched_rank_biserial(
    differences,
):
    differences = np.asarray(
        differences,
        dtype=float,
    )

    nonzero_differences = differences[
        ~np.isclose(
            differences,
            0.0,
        )
    ]

    if len(
        nonzero_differences
    ) == 0:
        return 0.0

    ranks = rankdata(
        np.abs(
            nonzero_differences
        ),
        method="average",
    )

    positive_rank_sum = float(
        ranks[
            nonzero_differences
            > 0
        ].sum()
    )

    negative_rank_sum = float(
        ranks[
            nonzero_differences
            < 0
        ].sum()
    )

    total_rank_sum = (
        positive_rank_sum
        + negative_rank_sum
    )

    if np.isclose(
        total_rank_sum,
        0.0,
    ):
        return 0.0

    return float(
        (
            positive_rank_sum
            - negative_rank_sum
        )
        / total_rank_sum
    )

In [ ]:
def holm_adjustment(
    p_values,
):
    p_values = np.asarray(
        p_values,
        dtype=float,
    )

    number_of_tests = len(
        p_values
    )

    order = np.argsort(
        p_values
    )

    sorted_p_values = p_values[
        order
    ]

    adjusted_sorted = np.empty(
        number_of_tests,
        dtype=float,
    )

    running_maximum = 0.0

    for index, p_value in enumerate(
        sorted_p_values
    ):
        multiplier = (
            number_of_tests
            - index
        )

        adjusted_value = min(
            1.0,
            float(
                p_value
                * multiplier
            ),
        )

        running_maximum = max(
            running_maximum,
            adjusted_value,
        )

        adjusted_sorted[
            index
        ] = running_maximum

    adjusted = np.empty(
        number_of_tests,
        dtype=float,
    )

    adjusted[
        order
    ] = adjusted_sorted

    return adjusted

In [ ]:
statistical_rows = []

for model in MODELS:
    pair_controlled_df = (
        results_by_model[
            model
        ][
            "pair_controlled"
        ]
    )

    max_cross_split_df = (
        results_by_model[
            model
        ][
            "max_cross_split"
        ]
    )

    for metric in INFERENTIAL_METRICS:
        pair_controlled_values = (
            pair_controlled_df[
                metric
            ]
            .astype(float)
            .to_numpy()
        )

        max_cross_split_values = (
            max_cross_split_df[
                metric
            ]
            .astype(float)
            .to_numpy()
        )

        differences = (
            pair_controlled_values
            - max_cross_split_values
        )

        permutation_result = (
            exact_paired_permutation_test(
                differences
            )
        )

        bootstrap_result = (
            paired_bootstrap_interval(
                differences
            )
        )

        wilcoxon_result = wilcoxon(
            pair_controlled_values,
            max_cross_split_values,
            alternative="two-sided",
            zero_method="wilcox",
            method="exact",
        )

        statistical_rows.append(
            {
                "model": model,
                "metric": metric,
                "condition_a": "pair_controlled",
                "condition_b": "max_cross_split",
                "difference_direction": (
                    "condition_a_minus_condition_b"
                ),
                "runs": int(
                    len(
                        differences
                    )
                ),
                "condition_a_mean": float(
                    pair_controlled_values.mean()
                ),
                "condition_a_std": float(
                    pair_controlled_values.std(
                        ddof=1
                    )
                ),
                "condition_b_mean": float(
                    max_cross_split_values.mean()
                ),
                "condition_b_std": float(
                    max_cross_split_values.std(
                        ddof=1
                    )
                ),
                "mean_difference": float(
                    differences.mean()
                ),
                "std_difference": float(
                    differences.std(
                        ddof=1
                    )
                ),
                "median_difference": float(
                    np.median(
                        differences
                    )
                ),
                "all_differences_positive": bool(
                    (
                        differences
                        > 0
                    ).all()
                ),
                "all_differences_negative": bool(
                    (
                        differences
                        < 0
                    ).all()
                ),
                **permutation_result,
                "wilcoxon_statistic": float(
                    wilcoxon_result.statistic
                ),
                "wilcoxon_p_value": float(
                    wilcoxon_result.pvalue
                ),
                "cohens_dz": paired_cohens_dz(
                    differences
                ),
                "rank_biserial": matched_rank_biserial(
                    differences
                ),
                **bootstrap_result,
            }
        )

statistical_analysis_df = pd.DataFrame(
    statistical_rows
)

In [ ]:
statistical_analysis_df[
    "permutation_p_value_holm"
] = holm_adjustment(
    statistical_analysis_df[
        "permutation_p_value"
    ].to_numpy()
)

statistical_analysis_df[
    "wilcoxon_p_value_holm"
] = holm_adjustment(
    statistical_analysis_df[
        "wilcoxon_p_value"
    ].to_numpy()
)

display(
    statistical_analysis_df
)

In [ ]:
statistical_analysis_df.to_csv(
    OUTPUT_PATH,
    index=False,
    encoding="utf-8",
)

print(
    "Statistical analysis saved to:",
    OUTPUT_PATH,
)